In [2]:
"""
Source reliability scorecard — matrix + scatter.
Edit the CONFIG block, then run:  python make_reliability_graphs.py
Inputs (same folder): employment_all_candidates.csv, contamination_removed.csv, employment_excluded_outliers.csv
Outputs: source_reliability_scorecard.csv/.xlsx, source_reliability_matrix.png, source_reliability_scatter.png
"""
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================ CONFIG (tweak here) ============================
EXCLUDE_SOURCES = ["xlsx_de"]          # not in candidate pool -> not assessable
FAMILY_COLORS   = {"archived": "#1f77b4", "wikipedia": "#2ca02c", "press": "#E08A1E"}
DISAGREE_THRESHOLD = None              # set e.g. 0.05 to redefine "agree within X%"; None = exact final-value match
MATRIX_FIGSIZE  = (11, 9)
SCATTER_FIGSIZE = (11, 7.5)
BUBBLE_SCALE    = 2.2                  # scatter bubble size = n_values * BUBBLE_SCALE
# columns shown in the matrix: (csv_column, header, colormap, higher_is_worse)
MATRIX_COLS = [
    ("n_values",          "Vol.",        "Blues",   False),
    ("contamination_pct", "Contam.\n%",  "Reds",    True),
    ("outlier_pct",       "Outlier\n%",  "Reds",    True),
    ("false_positive_pct","False-pos.\n%","Reds",   True),
    ("disagreement_pct",  "Disagree\n%", "Oranges", True),
    ("adoption_pct",      "Adoption\n%", "Greens",  True),
]
MATRIX_TITLE  = "Source reliability map\n(red = more errors · green = more often kept · label colour = family)"
SCATTER_TITLE = "Source reliability: false positives vs noise\n(size = number of values contributed)"
# ===========================================================================


def build_scorecard():
    num = lambda s: pd.to_numeric(s, errors="coerce")
    C   = pd.read_csv("employment_all_candidates.csv")
    C["emp"] = num(C["employees"]); C["yr"] = num(C["year"])
    con = pd.read_csv("contamination_removed.csv");  con["emp"] = num(con["emp"]);       con["yr"] = num(con["ref"])
    out = pd.read_csv("employment_excluded_outliers.csv"); out["emp"] = num(out["employees"]); out["yr"] = num(out["year"])

    keyset = lambda df: set(zip(df["source"], df["plant_id"], df["yr"].astype("Int64"), df["emp"].astype("Int64")))
    poolk, conk, outk = keyset(C), keyset(con), keyset(out)
    allk = poolk | conk | outk
    sources = sorted(set(C["source"]) | set(con["source"]) | set(out["source"]))

    # disagreement on cells observed by >=2 distinct sources
    C["multi"] = C.groupby(["plant_id", "yr"])["source"].transform("nunique") >= 2
    C["agree"] = C["matches_final_value"] == True
    fam = C.drop_duplicates("source").set_index("source")["family"].to_dict()

    rows = []
    for s in sources:
        n   = len({k for k in allk if k[0] == s})
        nc  = len({k for k in conk if k[0] == s})
        no  = len({k for k in outk if k[0] == s})
        cs  = C[C["source"] == s]; csm = cs[cs["multi"]]
        rows.append({
            "source": s, "family": fam.get(s, "?"), "n_values": n,
            "contamination_pct":  round(100*nc/n, 1) if n else 0,
            "outlier_pct":        round(100*no/n, 1) if n else 0,
            "false_positive_pct": round(100*(nc+no)/n, 1) if n else 0,
            "disagreement_pct":   round((1-csm["agree"].mean())*100, 1) if len(csm) else np.nan,
            "n_multisource":      len(csm),
            "adoption_pct":       round(cs["agree"].mean()*100, 1) if len(cs) else np.nan,
        })
    S = pd.DataFrame(rows)
    S = S[~S["source"].isin(EXCLUDE_SOURCES)].copy()
    return S


def draw_matrix(S):
    S = S.sort_values("false_positive_pct").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=MATRIX_FIGSIZE)
    for j, (col, lab, cmap, worse) in enumerate(MATRIX_COLS):
        vals = S[col].astype(float).values
        vmin, vmax = np.nanmin(vals), np.nanmax(vals); cm = plt.get_cmap(cmap)
        for i, v in enumerate(vals):
            if np.isnan(v):
                ax.add_patch(plt.Rectangle((j, i), 1, 1, color="#F2F2F2")); ax.text(j+.5, i+.5, "—", ha="center", va="center", color="#999"); continue
            norm = (v-vmin)/(vmax-vmin) if vmax > vmin else 0.4
            ax.add_patch(plt.Rectangle((j, i), 1, 1, color=cm(0.15+0.7*norm)))
            txt = f"{int(v)}" if col == "n_values" else f"{v:.0f}"
            ax.text(j+.5, i+.5, txt, ha="center", va="center", fontsize=9,
                    color="white" if norm > 0.6 else "#222",
                    fontweight="bold" if col == "false_positive_pct" else "normal")
    ax.set_xlim(0, len(MATRIX_COLS)); ax.set_ylim(0, len(S))
    ax.set_xticks([j+.5 for j in range(len(MATRIX_COLS))]); ax.set_xticklabels([m[1] for m in MATRIX_COLS], fontsize=9)
    ax.set_yticks([i+.5 for i in range(len(S))]); ax.set_yticklabels(S["source"], fontsize=9.5)
    for tick, f in zip(ax.get_yticklabels(), S["family"]): tick.set_color(FAMILY_COLORS.get(f, "#333"))
    ax.xaxis.tick_top(); ax.invert_yaxis()
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(length=0); ax.set_title(MATRIX_TITLE, fontsize=12, pad=30)
    plt.tight_layout(); plt.savefig("source_reliability_matrix.png", dpi=150, bbox_inches="tight", facecolor="white"); plt.close()


def draw_scatter(S):
    fig, ax = plt.subplots(figsize=SCATTER_FIGSIZE)
    for fam, c in FAMILY_COLORS.items():
        sub = S[S["family"] == fam]
        ax.scatter(sub["false_positive_pct"], sub["disagreement_pct"], s=sub["n_values"]*BUBBLE_SCALE,
                   color=c, alpha=.65, edgecolor="white", linewidth=1, label=fam)
    for r in S.itertuples():
        if not np.isnan(r.disagreement_pct):
            ax.annotate(r.source, (r.false_positive_pct, r.disagreement_pct), fontsize=8,
                        xytext=(4, 4), textcoords="offset points", color="#333")
    ax.set_xlabel("False-positive rate (contamination + outliers, %)")
    ax.set_ylabel("Disagreement vs consensus (multi-source cells, %)")
    ax.set_title(SCATTER_TITLE, fontsize=12, pad=10)
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    ax.grid(color="#EEE", lw=0.7); ax.set_axisbelow(True); ax.legend(title="Family", fontsize=9, frameon=False)
    plt.tight_layout(); plt.savefig("source_reliability_scatter.png", dpi=150, bbox_inches="tight", facecolor="white"); plt.close()


#if __name__ == "__main__":
    #S = build_scorecard()
    #S.sort_values("false_positive_pct", ascending=False).to_csv("source_reliability_scorecard.csv", index=False)
    #S.sort_values("false_positive_pct", ascending=False).to_excel("source_reliability_scorecard.xlsx", index=False)
    #draw_scatter(S)
    #print("Done: scorecard.csv/.xlsx, matrix.png, scatter.png")
